## Twitter Sentiment Analysis: Model Building and Evaluation

---

### Notebook Overview
Now that we have already **preprocessed** the Sentiment140 dataset and performed **feature engineering** in earlier steps, we can focus on **model building** and **evaluation**. In this notebook, we will import the finalized feature sets (including both text-based and temporal features) and explore various classification models for sentiment analysis.

---

## Table of Contents
1. [Introduction](#introduction)  
   1.1 [Purpose of This Notebook](#purpose)  
   1.2 [Preprocessed Data Overview](#data-overview)  
2. [Data Import](#data-import)  
   - Loading train/test sets  
   - Verifying shapes and class distributions  
3. [Baseline Models](#baseline-models)  
   - Logistic Regression  
   - Naive Bayes  
   - Evaluation metrics (Accuracy, F1-score, Precision, Recall)  
4. [Advanced Models](#advanced-models)  
   - Random Forest / Gradient Boosting  
   - (Optional) Neural Networks or Transformers  
   - Hyperparameter tuning strategies  
5. [Model Comparison and Selection](#model-comparison)  
   - Comparing performance across models  
   - Trade-offs in complexity vs. accuracy  
6. [Conclusion](#conclusion)  
   - Summary of results  
   - Recommendations and next steps  

---


## 1. Introduction <a name="introduction"></a>

### 1.1 Purpose of This Notebook <a name="purpose"></a>
We will build and evaluate **machine learning models** for **sentiment analysis** using the **processed** and **feature-engineered** Sentiment140 data from previous steps. Our objectives include:
- **Importing** the train/test splits prepared in the last notebook (including text-based TF-IDF features, n-gram features, and temporal features).
- **Training** multiple classification models to predict tweet sentiment.
- **Evaluating** and **comparing** these models using standard metrics.

### 1.2 Preprocessed Data Overview <a name="data-overview"></a>
From the previous notebook, we have:

- **X_train, X_test**: Feature matrices (sparse TF-IDF features augmented with temporal data).  
- **y_train, y_test**: Target labels indicating tweet sentiment (0 = negative, 1 = positive).

We also addressed:
- **Imbalanced data** (undersampling or other techniques).
- **n-grams** for richer linguistic context.
- **Temporal features** (day of week, hour of day, month) for potential time-related sentiment patterns.

With all preprocessing completed, we are now ready to focus entirely on model development and optimization.

---

# 2. Data Import <a name="data-import"></a>

In this section, we will:
- Load the train/test data (`X_train, X_test, y_train, y_test`) from the files we saved previously.
- Perform basic checks on shapes and distributions to confirm everything is in order.

In [1]:
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import time


In [2]:
# 1. Load the train/test datasets
X_train = joblib.load('./modeling_data/ML/X_train.joblib')
X_test = joblib.load('./modeling_data/ML/X_test.joblib')
y_train = joblib.load('./modeling_data/ML/y_train.joblib')
y_test = joblib.load('./modeling_data/ML/y_test.joblib')

# 2. Perform basic checks
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# Verify class distributions
print("\nClass Distribution in y_train:")
print(np.unique(y_train, return_counts=True))

print("\nClass Distribution in y_test:")
print(np.unique(y_test, return_counts=True))


Shape of X_train: (1173427, 10003)
Shape of X_test: (293357, 10003)
Shape of y_train: (1173427,)
Shape of y_test: (293357,)

Class Distribution in y_train:
(array([0, 1]), array([586713, 586714]))

Class Distribution in y_test:
(array([0, 1]), array([146679, 146678]))


#### Data Import Summary

##### Shapes are Consistent:
- **X_train**:  
  - Contains 1,277,126 samples and 10,003 features.  
  - Features include 10,000 TF-IDF features and 3 temporal features.  
- **X_test**:  
  - Contains 319,282 samples and the same number of features.  

This confirms that the dataset is properly prepared, combining both text-based and temporal features.

##### Class Distributions are Balanced:
- **y_train**:  
  - Contains an equal number of samples for both classes:  
    - Class 0 (Negative Sentiment): 638,563 samples  
    - Class 1 (Positive Sentiment): 638,563 samples  

- **y_test**:  
  - Also contains an equal number of samples for both classes:  
    - Class 0 (Negative Sentiment): 159,641 samples  
    - Class 1 (Positive Sentiment): 159,641 samples  

The balanced distribution ensures that the dataset is well-suited for training and testing sentiment analysis models without any class bias.


---

# 3. Baseline Models <a name="baseline-models"></a>

We will establish baseline performance using simpler models such as:
- **Logistic Regression**
- **Naive Bayes**

This gives us a quick benchmark to compare more advanced approaches against.



In [3]:
# Define a function to evaluate a model
def evaluate_model(model, X_train, y_train, X_test, y_test):
    start_time = time.time()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    end_time = time.time()

    # Print model results
    print(f"Model: {model.__class__.__name__}")
    print(f"Training Time: {end_time - start_time:.2f} seconds")
    print("\nClassification Report:")
    print(classification_report(y_test, predictions, target_names=['Negative', 'Positive']))
    print("Accuracy:", accuracy_score(y_test, predictions))
    print("Precision:", precision_score(y_test, predictions))
    print("Recall:", recall_score(y_test, predictions))
    print("F1 Score:", f1_score(y_test, predictions))
    print("-" * 50)

  # Return results for comparison table
    return {
        "Model": model.__class__.__name__,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions)
    }

**Logistic Regression**


In [4]:
# Logistic Regression
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
lg_results = evaluate_model(logistic_model, X_train, y_train, X_test, y_test)

Model: LogisticRegression
Training Time: 69.98 seconds

Classification Report:
              precision    recall  f1-score   support

    Negative       0.79      0.76      0.77    146679
    Positive       0.77      0.80      0.78    146678

    accuracy                           0.78    293357
   macro avg       0.78      0.78      0.78    293357
weighted avg       0.78      0.78      0.78    293357

Accuracy: 0.7763680430328916
Precision: 0.7651279300962746
Recall: 0.7975633701032193
F1 Score: 0.7810090328266138
--------------------------------------------------


Insights:
Logistic Regression performs reasonably well, achieving balanced precision and recall, making it a strong baseline for sentiment classification. It slightly favors recall over precision for positive sentiment.

**Naive Bayes**


In [5]:
# Naive Bayes (Multinomial)
naive_bayes_model = MultinomialNB()
nv_results = evaluate_model(naive_bayes_model, X_train, y_train, X_test, y_test)

Model: MultinomialNB
Training Time: 0.16 seconds

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.74      0.75    146679
    Positive       0.75      0.76      0.75    146678

    accuracy                           0.75    293357
   macro avg       0.75      0.75      0.75    293357
weighted avg       0.75      0.75      0.75    293357

Accuracy: 0.7487498167761465
Precision: 0.7454424367995479
Recall: 0.7554848034470064
F1 Score: 0.7504300245147834
--------------------------------------------------


Naive Bayes is computationally efficient and provides a competitive performance with slightly lower accuracy and F1 score compared to Logistic Regression.


---

## 4. Advanced Models <a name="advanced-models"></a>

After baseline experiments, we can explore more sophisticated methods:
- **Ensemble Models** (e.g., Random Forest, XGBoost, LightGBM)
- **Deep Learning Approaches** (optional, if resources permit, e.g., a neural network using embedding layers or transformer-based models)

We will also look at strategies for **hyperparameter tuning** (Grid Search, Randomized Search, Bayesian Optimization) to maximize performance.



**Ensemble Models** 

- Random Forest

In [6]:
# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
rf_results = evaluate_model(rf_model, X_train, y_train, X_test, y_test)

Model: RandomForestClassifier
Training Time: 4.75 seconds

Classification Report:
              precision    recall  f1-score   support

    Negative       0.73      0.68      0.71    146679
    Positive       0.70      0.75      0.73    146678

    accuracy                           0.72    293357
   macro avg       0.72      0.72      0.72    293357
weighted avg       0.72      0.72      0.72    293357

Accuracy: 0.7166864946123666
Precision: 0.702089373823694
Recall: 0.7527986473772481
F1 Score: 0.7265602895213028
--------------------------------------------------


- XGBoost

In [7]:
# XGBoost
xgb_model = XGBClassifier(n_estimators=100, max_depth=6, n_jobs=-1, tree_method='hist', random_state=42)
xg_results = evaluate_model(xgb_model, X_train, y_train, X_test, y_test)

Model: XGBClassifier
Training Time: 42.00 seconds

Classification Report:
              precision    recall  f1-score   support

    Negative       0.79      0.70      0.74    146679
    Positive       0.73      0.81      0.77    146678

    accuracy                           0.75    293357
   macro avg       0.76      0.75      0.75    293357
weighted avg       0.76      0.75      0.75    293357

Accuracy: 0.7531574157085054
Precision: 0.7272310036533202
Recall: 0.8102033024720817
F1 Score: 0.7664782273590656
--------------------------------------------------


**Deep Learning Approaches**

We will use a different notebook for this. because we need to work the data processing in a different way. For example we will need to use embedings.

---

# 5. Model Comparison and Selection <a name="model-comparison"></a>

We will compare all tested models using:
- **Accuracy**
- **Precision, Recall, F1-score** (especially relevant for imbalanced classes, if that remains a concern)
- **Confusion Matrix** to visualize classification performance



In [8]:
comparison_table = pd.DataFrame([lg_results, nv_results, rf_results, xg_results])
comparison_table

,Model,Accuracy,Precision,Recall,F1 Score
0,LogisticRegression,0.776368,0.765128,0.797563,0.781009
1,MultinomialNB,0.748750,0.745442,0.755485,0.750430
2,RandomForestClassifier,0.716686,0.702089,0.752799,0.726560
3,XGBClassifier,0.753157,0.727231,0.810203,0.766478


#### Analysis

1. **Logistic Regression**:
   - **Accuracy**: Achieved the highest accuracy (0.779) among all models.
   - **Precision/Recall Balance**: Well-balanced with a slight favor toward recall.
   - **F1-Score**: Outperforms other models slightly in F1-Score, making it a strong baseline.

2. **Multinomial Naive Bayes**:
   - **Performance**: Slightly lower accuracy (0.753) and F1-Score compared to Logistic Regression.
   - **Training Speed**: Extremely fast to train, making it an efficient choice for quick experiments.

3. **Random Forest**:
   - **Accuracy**: Lower accuracy (0.714) compared to Logistic Regression and XGBoost.
   - **Precision/Recall**: Balanced but slightly lower overall performance than other models.
   - **Use Case**: May benefit from hyperparameter tuning and feature importance analysis.

4. **XGBoost**:
   - **Recall**: Achieved the highest recall (0.812), suggesting strong capability in identifying positive samples.
   - **F1-Score**: Competitive F1-Score (0.771), slightly trailing Logistic Regression.
   - **Computational Cost**: Slightly slower to train but effective for imbalanced data.

---

#### Recommendations

- **Best Performing Model**: Logistic Regression offers the best trade-off between accuracy, precision, recall, and F1-Score.
- **For High Recall Needs**: If recall is critical (e.g., minimizing false negatives), XGBoost is a strong contender.
- **Future Improvements**:
  - **Hyperparameter Tuning**: Fine-tune parameters for Random Forest and XGBoost to explore potential performance gains.
  - **Deep Learning Models**: Explore transformer-based models (e.g., BERT) for potentially superior performance.

---

# 6. Conclusion <a name="conclusion"></a>

### Summary of Results
In this notebook, we evaluated multiple machine learning models for sentiment analysis on tweets. Below is a summary of their performance:

- **Logistic Regression**:  
  - Achieved the highest overall performance with an accuracy of **0.779** and an F1-Score of **0.784**.  
  - Its balance between precision and recall makes it a strong baseline model for sentiment analysis.  

- **Multinomial Naive Bayes**:  
  - Slightly lower accuracy (**0.753**) and F1-Score (**0.754**) compared to Logistic Regression.  
  - Extremely fast to train, making it an efficient choice for scenarios requiring rapid iterations.

- **Random Forest**:  
  - Delivered moderate performance with an accuracy of **0.714** and F1-Score of **0.717**.  
  - This model may benefit from additional hyperparameter tuning and feature selection.  

- **XGBoost**:  
  - Demonstrated strong recall (**0.812**) and competitive overall performance with an F1-Score of **0.771**.  
  - Slower to train compared to Logistic Regression and Naive Bayes but is well-suited for imbalanced datasets.

### Recommendations & Next Steps
1. **Model Selection**:  
   - For general use cases, **Logistic Regression** is the most suitable model, balancing simplicity and performance.
   - For applications prioritizing recall (e.g., minimizing false negatives), **XGBoost** is a strong contender.

2. **Possible Enhancements**:
   - **Hyperparameter Tuning**:
     - Optimize parameters for Random Forest and XGBoost to explore potential performance improvements.
     - Techniques such as Grid Search, Randomized Search, or Bayesian Optimization could be employed.
   - **Transformer-based Models**:
     - Consider using pre-trained models like **BERT** or **RoBERTa** for potentially superior text understanding and sentiment classification.
   - **Domain-specific Fine-Tuning**:
     - Fine-tune transformer-based models on the Sentiment140 dataset to capture domain-specific nuances in tweets.

3. **Deployment Considerations**:
   - **Inference Pipeline**:
     - Evaluate the computational cost of deploying the selected model in a real-time or batch processing pipeline.
     - Logistic Regression is lightweight and ideal for real-time inference.
   - **Scalability**:
     - Consider using cloud-based solutions or GPUs for models like XGBoost or transformers in high-throughput environments.
   - **Monitoring and Feedback**:
     - Implement monitoring to track model performance on live data and retrain periodically to address potential data drift.
